In [ ]:
import json
import random
import argparse
#从musique数据集生成qa.jsonl和mcorpus.json
def main(
    input_path: str,
    qa_output_path: str = "qa.jsonl",
    corpus_output_path: str = "mcorpus.json",
    sample_size: int = 7500,
    seed: int = 42
):
    random.seed(seed)

    # Step 1: 读取所有样本到内存
    all_samples = []
    with open(input_path, "r", encoding="utf-8") as f_in:
        for line in f_in:
            line = line.strip()
            if line:
                all_samples.append(json.loads(line))

    total = len(all_samples)
    if total == 0:
        raise ValueError("Input file is empty!")

    # Step 2: 随机抽样
    if total <= sample_size:
        print(f"⚠️  Total samples ({total}) <= {sample_size}. Using all data.")
        sampled_samples = all_samples
    else:
        sampled_samples = random.sample(all_samples, sample_size)
        print(f"✅ Randomly sampled {sample_size} out of {total} samples (seed={seed}).")

    # Step 3: 处理 QA 和段落
    seen_paragraphs = {}
    qa_records = []

    for sample in sampled_samples:
        # 收集 QA
        qa_records.append({
            "question": sample["question"],
            "answer": sample["answer"]
        })

        # 收集段落（去重）
        for para in sample.get("paragraphs", []):
            text = para["paragraph_text"]
            if text not in seen_paragraphs:
                seen_paragraphs[text] = {
                    "idx": para.get("idx"),
                    "title": para["title"],
                    "paragraph_text": text
                }

    # Step 4: 写入 QA 文件（JSONL）
    with open(qa_output_path, "w", encoding="utf-8") as f_qa:
        for record in qa_records:
            f_qa.write(json.dumps(record, ensure_ascii=False) + "\n")

    # Step 5: 写入语料库（JSON 数组）
    corpus_list = list(seen_paragraphs.values())
    try:
        corpus_list.sort(key=lambda x: x.get("idx", 0))
    except Exception:
        pass

    with open(corpus_output_path, "w", encoding="utf-8") as f_corpus:
        json.dump(corpus_list, f_corpus, ensure_ascii=False, indent=2)

    print(f"✅ Final QA pairs: {len(qa_records)}")
    print(f"✅ Unique paragraphs: {len(corpus_list)}")
    print(f"✅ QA saved to: {qa_output_path}")
    print(f"✅ Corpus saved to: {corpus_output_path}")


if __name__ == "__main__":
    main(
        input_path="musique.jsonl",
        qa_output_path="qa.jsonl",
        corpus_output_path="mcorpus.json",
        sample_size=7500,
        seed=42
    )


In [ ]:
from typing import List,Dict
from transformers import AutoTokenizer
import re
import aiofiles
import aiohttp
from autogen_core.memory import Memory, MemoryContent, MemoryMimeType
from dateutil import parser
class TokenBasedDocumentIndexer:
    """
    Splits text into chunks of up to `max_tokens` using Qwen2.5's tokenizer,
    with `overlap_tokens` token overlap between chunks.
    """

    def __init__(
        self,
        model_name: str = "",#模型地址
        max_tokens: int = 1200,
        overlap_tokens: int = 100,
        trust_remote_code: bool = True,
        memory: Memory = None,
    ):
        self.max_tokens = max_tokens
        self.overlap_tokens = overlap_tokens
        # 加载 模型 的 tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=trust_remote_code,
        )
        self.memory = memory

    async def index_documents(self, documents: List[Dict]) -> int:
        total_chunks = 0
        for doc in documents:
            #info="    source: "+doc['source']+"      published at: "+doc['published_at']#处理multihop-rag数据集时添加metadata
            # 过滤掉空值，格式化
            full_text = doc["paragraph_text"]
            await self.memory.add(
                MemoryContent(
                    content=full_text, mime_type=MemoryMimeType.TEXT
                )
            )
            total_chunks += 1    
        return total_chunks

In [ ]:
# 导入必要的库
import os
import yaml
from autogen_ext.memory.chromadb import ChromaDBVectorMemory, PersistentChromaDBVectorMemoryConfig,CustomEmbeddingFunctionConfig
from pathlib import Path
import json
with open('settings.yaml', 'r') as f:
    config = yaml.safe_load(f)
# Initialize vector memory
vector_store_config = PersistentChromaDBVectorMemoryConfig(
        collection_name=config['vector_store']['collection_name'],
        persistence_path=os.path.expandvars(config['vector_store']['persistence_path'].replace('${HOME}', str(Path.home()))),
        k=config['vector_store']['k'],
        score_threshold=config['vector_store']['score_threshold'],
)
    
# Check if custom embedding function is enabled in config
if config['vector_store'].get('embedding', {}).get('use_custom', False):
        def create_openai_embedding_function(api_key, model, api_base):
            from chromadb.utils import embedding_functions
            return embedding_functions.OpenAIEmbeddingFunction(
                api_key=api_key,
                model_name=model,
                api_base=api_base
            )
        
        embedding_config = config['vector_store']['embedding']
        params = {
            "api_key": embedding_config['api_key'],
            "model": embedding_config['model'],
            "api_base": embedding_config['api_base']
        }
        
        vector_store_config.embedding_function_config = CustomEmbeddingFunctionConfig(
            function=create_openai_embedding_function,
            params=params
        )
    
rag_memory = ChromaDBVectorMemory(config=vector_store_config)

await rag_memory.clear()  # Clear existing memory


# Index AutoGen documentation
async def index_autogen_docs() -> None:
    indexer = TokenBasedDocumentIndexer(memory=rag_memory)
    input_file = "../mcorpus.json"
    with open(input_file, 'r', encoding='utf-8') as f:
        sources = json.load(f)
    #sources = [line['body'] for line in lines]
    chunks: int = await indexer.index_documents(sources)
    print(f"Indexed {chunks} chunks from {len(sources)} AutoGen documents")


await index_autogen_docs()


In [ ]:
import json
import pickle
from rank_bm25 import BM25Okapi
import os

def build_bm25(chunks_path="../mcorpus.json", output_path="../mbm25.pkl"):
    with open(chunks_path, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    # 直接使用 "lower" 字段（已包含 source + published_at + 正文，且全小写）
    corpus = [chunk["paragraph_text"].lower() for chunk in chunks]

    # 英文/空格分词
    tokenized_corpus = [text.split() for text in corpus]

    # 构建 BM25
    bm25 = BM25Okapi(tokenized_corpus)

    # 保存模型和原始 chunks（用于返回 original）
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    with open(output_path, "wb") as f:
        pickle.dump({"bm25": bm25, "chunks": chunks}, f)

    print(f"✅ Built BM25 index with {len(chunks)} chunks.")
build_bm25()